# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [1]:
from pathlib import Path

import ibis
from hamilton import driver, base

from iacs.audit_system import (
    AuditRunner,
    RequirementCoverageAudit,
    TraceabilityAudit,
    TodoAudit,
)
from iacs.transforms import manifest_to_registry
from iacs.transform_system import TransformSystem

ibis.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [2]:
COMPONENTS_DIR = str(Path("../components"))

# Build registry from YAML manifests using Hamilton DAG
build_driver = driver.Driver(
    {"input_dir": COMPONENTS_DIR},
    manifest_to_registry,
    adapter=base.DictResult(),
)
result = build_driver.execute(["registry"])
reg = result["registry"]

# Create TransformSystem for post-registry transforms
ts = TransformSystem(reg, [manifest_to_registry])

print(f"Component types: {reg.component_types}")
print(f"Number of component types: {len(reg.component_types)}")
print(f"Available transforms: {ts.outputs}")

Component types: ['description', 'file_info', 'requirement', 'parent', 'id', 'todo', 'system', 'input', 'output', 'weight']
Number of component types: 10
Available transforms: ['complete_schema', 'component_first_data', 'schema', 'parent', 'components_database', 'conn', 'components', 'data_models', 'flattened_entity_first_data', 'raw_entity_first_data', 'validated_components']


## Run All Audits

In [3]:
runner = AuditRunner(audits=[
    RequirementCoverageAudit(),
    TraceabilityAudit(),
    TodoAudit(),
])

results = runner.run(reg)

print(f"All audits passed: {runner.all_passed}")
print()
for name, result in results.items():
    status = "PASS" if result.passed else "FAIL"
    print(f"{name}: {status}")

All audits passed: False

requirement_coverage: FAIL
traceability: FAIL
todo: FAIL


## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [4]:
reg.view(["requirement", "description"])

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id                                                                        ┃ priority ┃ requirement.value ┃ description.value                                                                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string                                                                           │ float64  │ string            │ string                                                                           │
├──────────────────────────────────────────────────────────────────────────────────┼──────────┼───────────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ iacs.be_a_powerful_tool_for_solutions_architecture                               │     NULL │ NULL              │ iacs (short for Infrastructure-as-Code Sketch) is software designed to help tec… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │      1.0 │ NULL              │ Provide a means to comprehensively and accurately document infrastructure.\n     │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ If infrastructure is defined elsewhere in an existing format, we don't want to … │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ We don't want to duplicate other IaC code if possible.\n                         │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ Code such as Python also provides clear definitions.\n                           │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ constraint        │ Version control is essential for reliability.\n                                  │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ A source of truth is not *a* source of truth if it is distributed.\n             │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │      1.0 │ NULL              │ This is one of the key features iacs provides that other modeling tools do not-… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ Users need to be able to describe the infrastructure they want to build.\n       │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │     NULL │ NULL              │ One potential solution for allowing users to define infrastructure is via an EC… │
│ …                                                                                │        … │ …                 │ …                                                                                │
└──────────────────────────────────────────────────────────────────────────────────┴──────────┴───────────────────┴──────────────────────────────────────────────────────────────────────────────────┘

In [5]:
results["requirement_coverage"].view()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id                                                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string                                                                           │
├──────────────────────────────────────────────────────────────────────────────────┤
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.help_improve_the_quality_of_… │
│ ecs_framework.requirements.describe_hierarchical_structure.define_entity_ids_by… │
│ ecs_framework.requirements.describe_hierarchical_structure.enable_flexible_synt… │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │
│ ecs_framework.requirements.describe_systems.describe_entities                    │
│ ecs_framework.ecs_data_formats.entity_first.entity_item.entity_value.components… │
│ ecs_framework.ecs_data_formats.component_first.component_table.table_index       │
│ iacs.be_a_powerful_tool_for_solutions_architecture.provide_quality_end_to_end_d… │
│ ecs_framework.ecs_data_formats.entity_first.component_item.string_component_item │
│ …                                                                                │
└──────────────────────────────────────────────────────────────────────────────────┘

## Traceability Audit

Checks that all entities trace back to requirements.

In [6]:
trace_result = results["traceability"]
print(f"Passed: {trace_result.passed}")
print(f"Orphan entities: {len(trace_result.results)}")
print()
if not trace_result.results.empty:
    print("Entities not tracing to requirements:")
    trace_result.view()

Passed: False
Orphan entities: 80

Entities not tracing to requirements:


## Todo Audit

Reports outstanding todos.

In [7]:
todo_result = results["todo"]
print(f"Passed: {todo_result.passed}")
print(f"Entities with todos: {len(todo_result.results)}")
print()
if todo_result.messages:
    print("Outstanding todos:")
    for msg in todo_result.messages:
        print(f"  - {msg}")

Passed: False
Entities with todos: 7

Outstanding todos:
  - iacs.iacs_meta_discussion.syntax_challenges: Have files be the topmost entities.

  - iacs.iacs_meta_discussion.syntax_challenges: Notation for multiple components of the same type. Maybe "component_type s"?

  - iacs.iacs_meta_discussion.syntax_challenges: Binary relationship notation going both ways. Maybe "component_type of:" and "component_type:"?

  - iacs.iacs_meta_discussion.syntax_challenges: Notation for indicating that a component is a row-wise subset of another component, i.e. that it has the same fields but additional constraints. Maybe "subset of:" and "subset s", i.e. we use a "subset" relation?

  - iacs.iacs_meta_discussion.syntax_challenges: Notation for defining an entity in-line vs linking to an existing entity. Maybe "(entity_name)"?

  - iacs.iacs_meta_discussion.syntax_challenges: Notation for indicating that a component must be joined with another component to be interpreted correctly. Maybe: "required: